# Claim records and what a coding agent says: replication

This notebook checks the replication package of the paper from the public repository, with no setup:

- **(a)** clone the repository at the version the paper cites (`REF` below);
- **(b)** install what the package needs (`pytest` only);
- **(c)** verify the SHA-256 seal of the code, documents and frozen data (`make_provenance.py --verify`);
- **(d)** rerun every analysis that needs only the frozen scores, run the tests that lock the published numbers and
  verify the seal again (`run_all.py`);
- **(e)** check every number the paper prints against the output that prints it
  (`posthoc/verify_paper_numbers.py`, `posthoc/test_paper_numbers.py`) and show them side by side: paper value,
  value in the output, match;
- **(f)** optional: download the public raw trial archive from Zenodo
  ([10.5281/zenodo.24450523](https://doi.org/10.5281/zenodo.24450523)), check its SHA-256 and rerun the post hoc
  scripts that read it.

Steps (a) to (e) take about a minute and use no model and no data outside the repository. Step (f) downloads
about 0.5 GB, unpacks about 2.5 GB and took about 4 minutes more on a recent laptop; expect longer on Colab's free
machine.

**What this notebook cannot reproduce, and why**

- **The trials themselves.** Each trial was a session of a coding agent driven by a hosted language model. Those
  calls are not deterministic, the model and agent versions used may no longer be served, and running them again
  costs money and would produce different transcripts. What reruns is everything downstream of the saved
  transcripts: the scores, the registered analyses, the post hoc analyses and every number the paper prints.
- **The hidden checks and the container runs** happened inside the trials; their results are in the frozen scores
  and in the raw archive, not recomputed here.
- **The raters' answers** (the author, a second rater, and the model raters of the audits) are frozen in
  `runs/*-audit/`; the notebook reruns the comparisons, not the reading.
- **Two post hoc checks need files that are not published** (`audit_assist.py`, `keys_count.py`); the reason is in
  `posthoc/README.md`, "Excluded".
- **Figures with no script** (quoted from cited works, read by hand, or stated by design) are listed under
  `unscripted` in `posthoc/paper_numbers.json`, and printed at the end of step (e). They are not counted as matches.
- **`posthoc/prereg_chronology.py`** reads the live public ledger; step (f) reruns it only if a clone of that ledger
  and the OpenTimestamps client are available, and its output can change if the ledger history is rewritten.

## Settings

`REF` is the version checked. The paper pins a release tag or a commit: set `REF` to it so that this notebook checks
exactly the version the paper cites. `"main"` checks the latest version on GitHub.

In [ ]:
import html
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REF = "main"  # the paper's tag or commit, e.g. "v1.2.0"; "main" = latest
REPO_URL = "https://github.com/ulissesflores/eilikrineia-study.git"
RAW_RECORD = "24450523"  # Zenodo record of the raw trial archive (10.5281/zenodo.24450523)
LEDGER_URL = "https://github.com/ulissesflores/eilikrineia.git"  # the public ledger
RUN_RAW = False  # step (f): set to True to download the raw archive and rerun every post hoc script

# Overrides for automated test runs of this notebook only; a reader leaves them unset.
REPO_URL = os.environ.get("EILIKRINEIA_NB_REPO", REPO_URL)
REF = os.environ.get("EILIKRINEIA_NB_REF", REF)
RUN_RAW = RUN_RAW or os.environ.get("EILIKRINEIA_NB_RUN_RAW") == "1"
DEFAULT_WORK = "/content" if Path("/content").is_dir() else Path.cwd()
WORK = Path(os.environ.get("EILIKRINEIA_NB_WORKDIR", DEFAULT_WORK))
PKG = WORK / "eilikrineia-study"

# Nothing may write bytecode into the sealed folders; the package's Python runs every script.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


def sh(cmd, cwd=None, env=None):
    """Run a command, print its output as it comes, and stop the notebook if it fails."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    lines = []
    with subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
    if proc.returncode:
        raise RuntimeError(f"exit code {proc.returncode}: {' '.join(str(c) for c in cmd)}")
    return "".join(lines)


print(f"REF={REF}  WORK={WORK}  RUN_RAW={RUN_RAW}  Python {sys.version.split()[0]}")

## (a) Clone the repository at `REF`

In [ ]:
if PKG.exists():
    shutil.rmtree(PKG)  # start from a clean checkout every time
sh(["git", "clone", "--quiet", REPO_URL, str(PKG)])
sh(["git", "-c", "advice.detachedHead=false", "checkout", "--quiet", REF], cwd=PKG)
COMMIT = sh(["git", "rev-parse", "HEAD"], cwd=PKG).strip()
assert sh(["git", "status", "--porcelain"], cwd=PKG) == "", "the checkout is not clean"
print(f"checked out {REF} = commit {COMMIT}")

## (b) Install what the package needs

In [ ]:
sh([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], cwd=PKG)
sh([sys.executable, "-m", "pytest", "--version"], cwd=PKG)

## (c) Verify the seal

One SHA-256 `chain_hash` covers the code, the dated documents, the frozen data and the tests
(`output/hash-chain.md` lists every file). This recomputes it from the files and compares it with the committed one.

In [ ]:
sh([sys.executable, "make_provenance.py", "--verify"], cwd=PKG)

## (d) Rerun the analyses, run the tests, verify the seal again

`run_all.py` reruns every analysis that needs only the frozen scores and compares its output with the frozen file
byte for byte, runs the tests that lock the published numbers, and verifies the seal.

In [ ]:
RUN_ALL = sh([sys.executable, "run_all.py"], cwd=PKG)

## (e) Check every number the paper prints

`posthoc/paper_numbers.json` maps each number the paper prints to the committed output that prints it.
The verifier checks the SHA-256 of every script and output it names, checks every printed value against its
output line, and reruns the post hoc scripts that need only the package (the others need the raw archive of step
(f) and are skipped here, with the reason). The tests check the same, plus a test that the checker rejects a changed
value.

In [ ]:
PYTEST = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"]
sh([*PYTEST, "posthoc/test_paper_numbers.py"], cwd=PKG)
VERIFY = sh([sys.executable, "posthoc/verify_paper_numbers.py", "--rerun"], cwd=PKG)

In [ ]:
from IPython.display import HTML, display

sys.path.insert(0, str(PKG / "posthoc"))
import verify_paper_numbers as V  # noqa: E402  (the package's own checker, imported unchanged)


def rerun_status(run_all_text, verify_text):
    """Map each output file to what happened to it in this session."""
    status = {}
    for line in run_all_text.splitlines():
        m = re.match(r"\s+(identical|DIFFERENT): (\S+) -> (\S+)(.*)", line)
        if m:
            status[m[3]] = f"rerun here: {m[1]}" + (" (partly; rest needs raw)" if m[4] else "")
    for line in verify_text.splitlines():
        m = re.match(r"\s+(same|DIFFERENT|skipped \(.*\)): posthoc/\S+\.py -> (\S+)", line)
        if m:
            verdict = "skipped (inputs absent; see the verifier output)" if m[1][0] == "s" else m[1]
            verdict = "same" if m[1] == "same" else verdict
            status[m[2]] = f"rerun here: {verdict}"
    return status


def kind(label):
    """Classify a source label: full, partly, skipped, different or sealed."""
    for word, k in (("DIFFERENT", "different"), ("skipped", "skipped"), ("partly", "partly")):
        if word in label:
            return k
    return "full" if label.startswith("rerun here") else "sealed"


def paper_table(status):
    """Build one row per printed value: paper value, value in the output, match, source check."""
    man, read, rows = V.load(), V.reader(), []
    sealed = "sealed frozen output (seal verified)"
    for e in man["numbers"]:
        if e["kind"] == "derived":
            got, ok = "(error)", False
            try:
                got = V.derived_value(e, read)
                V.check_entry(e, read)
                ok = True
            except AssertionError:
                pass
            files = sorted({t["file"] for t in e["terms"]})
            labels = [status.get(f, sealed) for f in files]
            kinds = {kind(x) for x in labels}
            for k in ("different", "skipped"):
                if k in kinds:
                    kinds = {k}
            category = kinds.pop() if len(kinds) == 1 else "partly"
            check = "; ".join(f"{f}: {x}" for f, x in zip(files, labels, strict=True))
            rows.append(
                (e["where"], e["id"], e["printed"], f"{got} (derived)", ok, check, category)
            )
            continue
        line = V.anchored_line(read(e["output"]), e["anchor"], e.get("nth", 1), e["count"])
        for v in e["values"]:
            got = V.locate(line, v["in_output"]) or v["in_output"]
            try:
                V.check_value(line, v["printed"], v["in_output"], v["rule"])
                ok = True
            except AssertionError:
                ok = False
            check = status.get(e["output"], sealed)
            row = (
                e["where"],
                e["id"],
                v["printed"],
                got,
                ok,
                f"{e['output']}: {check}",
                kind(check),
            )
            rows.append(row)
    return man, rows


def show(status):
    """Display the summary, the side-by-side table and the figures that have no script."""
    man, rows = paper_table(status)
    matched = sum(r[4] for r in rows)
    print(
        f"{matched} of {len(rows)} printed values match their output "
        f"({len(man['numbers'])} number groups); {len(man['unscripted'])} groups have no script."
    )
    n = {
        k: sum(r[6] == k for r in rows)
        for k in ("full", "partly", "skipped", "different", "sealed")
    }
    print(f"Source of the {len(rows)} values in this session:")
    print(f"  {n['full']}: outputs that their script reprinted in full")
    print(f"  {n['partly']}: an output only partly rerun, or a mix of rerun and sealed outputs")
    print(f"  {n['skipped']}: script skipped (inputs absent)")
    print(f"  {n['sealed']}: sealed frozen outputs only (checked by hash)")
    print(f"  {n['different']}: a rerun that printed something else")
    head = ("Section", "Id", "Paper value", "Value in the output", "Match", "Source check")
    cells = "".join(
        "<tr>"
        + "".join(
            f"<td>{'yes' if c is True else 'NO' if c is False else html.escape(str(c))}</td>"
            for c in r[:6]
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            "<div style='max-height:420px;overflow:auto'><table><tr>"
            + "".join(f"<th>{h}</th>" for h in head)
            + f"</tr>{cells}</table></div>"
        )
    )
    print("Number groups with no script (not counted as matches):")
    for u in man["unscripted"]:
        print(f"  [{u['where']}] {u['id']}: {u['note']}")
    assert matched == len(rows), "some printed values do not match their output"
    assert n["different"] == 0, "a rerun printed something other than the committed output"


STATUS = rerun_status(RUN_ALL, VERIFY)
show(STATUS)

## (f) Optional: the raw trial archive

Set `RUN_RAW = True` in the settings cell and run this cell and the next. It downloads the public raw archive
(the transcripts, tool outputs and signed ledgers of every trial, with private keys and personal machine details
removed; [10.5281/zenodo.24450523](https://doi.org/10.5281/zenodo.24450523)), checks its SHA-256 against the
record's `SHA256SUMS`, unpacks it to `raw/` in the checkout, and reruns every post hoc script. `prereg_chronology.py`
also needs a full clone of the public ledger and the OpenTimestamps client (`ots`, used offline: `ots info`); this
cell clones the ledger and installs the client. `ledger_audit.py` needs OpenSSH's `ssh-keygen`.

In [ ]:
import hashlib
import json
import tarfile
import urllib.request


def fetch(url, dest=None):
    """Download a URL to a file (or return its bytes when no destination is given)."""
    with urllib.request.urlopen(url) as r:
        if dest is None:
            return r.read()
        with open(dest, "wb") as f:
            shutil.copyfileobj(r, f, 1 << 20)
    return dest


def sha256(path):
    """Return the SHA-256 of a file, read in chunks."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


if RUN_RAW:
    record = json.loads(fetch(f"https://zenodo.org/api/records/{RAW_RECORD}"))
    links = {f["key"]: f["links"]["self"] for f in record["files"]}
    sums = fetch(links["SHA256SUMS"]).decode()
    name = "eilikrineia-raw-trials.tar.gz"
    want = dict(line.split()[::-1] for line in sums.splitlines() if line.strip())[name]
    tarball = os.environ.get("EILIKRINEIA_NB_RAW_TARBALL")  # automated test runs only: a local copy
    if not tarball:
        free = shutil.disk_usage(WORK).free / 1e9
        assert free > 5, f"{free:.1f} GB free; the archive needs about 3 GB"
        tarball = fetch(links[name], WORK / name)
        print(f"downloaded {name} from record {RAW_RECORD}")
    got = sha256(tarball)
    assert got == want, f"SHA-256 {got} differs from the record's {want}"
    print(f"{name}: SHA-256 {got} equals the record's SHA256SUMS")
    raw = PKG / "raw"
    shutil.rmtree(raw, ignore_errors=True)
    raw.mkdir()
    with tarfile.open(tarball) as t:
        t.extractall(raw, filter="data")
    print("unpacked to raw/:", sorted(p.name for p in raw.iterdir()))
    ledger = WORK / "eilikrineia"
    if not (ledger / ".git").exists():
        sh(["git", "clone", "--quiet", LEDGER_URL, str(ledger)])
    if not shutil.which("ots"):
        sh([sys.executable, "-m", "pip", "install", "--quiet", "opentimestamps-client"])
    os.environ["EILIKRINEIA_RAW"] = str(raw)
    os.environ["EILIKRINEIA_CHAIN"] = str(ledger)
    for tool in ("ssh-keygen", "openssl", "ots", "git"):
        print(f"{tool}: {shutil.which(tool) or 'absent (the scripts that need it are skipped)'}")
else:
    print("RUN_RAW is False: step (f) skipped")

In [ ]:
if RUN_RAW:
    VERIFY_RAW = sh([sys.executable, "posthoc/verify_paper_numbers.py", "--rerun"], cwd=PKG)
    show(rerun_status(RUN_ALL, VERIFY_RAW))
else:
    print("RUN_RAW is False: step (f) skipped")